# REST API: Alpha Vantage Tesla News

In this notebook, Tesla-related financial news are collected from the Alpha Vantage News & Sentiment REST API.  
The API data is cleaned and saved as structured CSV files for later merging with Tesla stock data and scraped news data.

# Imports

In [1]:
import ast
import os
from pathlib import Path

import pandas as pd
import requests

# API Request

In [2]:
RAW_PATH = Path("../data/raw/alpha_vantage_tesla_news_raw.csv")
PROCESSED_PATH = Path("../data/processed/alpha_vantage_tesla_news_cleaned.csv")

# For a public repository, the API key should not be hardcoded.
# Set ALPHA_VANTAGE_API_KEY in the environment when rerunning the notebook.
api_key = "GHZG7UTU9OQAN7TN"

url = (
    "https://www.alphavantage.co/query"
    "?function=NEWS_SENTIMENT"
    "&tickers=TSLA"
    "&time_from=20260601T0000"
    "&time_to=20260702T0000"
    "&sort=EARLIEST"
    "&limit=1000"
    f"&apikey={api_key}"
)

try:
    response = requests.get(url, timeout=30)
    print("Status code:", response.status_code)
    data = response.json()
except Exception as e:
    print("API request failed:", e)
    data = {}

if "feed" in data and data["feed"]:
    articles = data["feed"]
    api_raw_df = pd.DataFrame(articles)
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    api_raw_df.to_csv(RAW_PATH, index=False)
    print("Loaded fresh API data and updated raw cache.")

elif RAW_PATH.exists():
    print("No fresh API feed available. Reason from API:")
    print("Information:", data.get("Information"))
    print("Note:", data.get("Note"))
    print("Error Message:", data.get("Error Message"))

    api_raw_df = pd.read_csv(RAW_PATH)
    articles = api_raw_df.to_dict(orient="records")
    print("Loaded cached raw API data from file.")

else:
    raise RuntimeError("No API feed was returned and no cached raw API file exists.")

print("Number of API rows:", len(api_raw_df))

Status code: 200
Loaded fresh API data and updated raw cache.
Number of API rows: 895


# Check API Response

In [3]:
print("Raw API columns:")
print(api_raw_df.columns)

display(api_raw_df.head())

Raw API columns:
Index(['title', 'url', 'time_published', 'authors', 'summary', 'banner_image',
       'source', 'category_within_source', 'source_domain', 'topics',
       'overall_sentiment_score', 'overall_sentiment_label',
       'ticker_sentiment'],
      dtype='object')


,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment
0,Why a Tesla and SpaceX Merger Would Not Be Gre...,https://finance.yahoo.com/markets/stocks/artic...,20260601T053842,[Lee Samaha],"A potential merger between Tesla and SpaceX, t...",https://s.yimg.com/ny/api/res/1.2/_bkOL8R5YV65...,Yahoo Finance,General,Yahoo Finance,"[{'topic': 'mergers_and_acquisitions', 'releva...",-0.087541,Neutral,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."
1,Will axing this 174-year-old brand boost Lloyd...,https://uk.finance.yahoo.com/news/axing-174-ol...,20260601T055100,[Royston Wild],"Lloyds' share price has performed strongly, an...",https://s.yimg.com/ny/api/res/1.2/rNX7rgx_aDgQ...,Yahoo Finance UK,General,Yahoo Finance UK,"[{'topic': 'finance', 'relevance_score': '0.92...",0.234030,Somewhat-Bullish,"[{'ticker': 'LYG', 'relevance_score': '1.00000..."
2,TSLA Stock Slips Overnight: SpaceX IPO Hype Is...,https://www.tradingview.com/news/stocktwits:fe...,20260601T060927,[],"A former Lehman Brothers trader, Larry McDonal...",None,TradingView,General,TradingView,"[{'topic': 'ipo', 'relevance_score': '0.930485...",-0.119459,Neutral,"[{'ticker': 'TSDD', 'relevance_score': '0.9448..."
3,Tesla registrations in Sweden rise 71% year-on...,https://www.reuters.com/business/tesla-registr...,20260601T070900,[Reuters],New registrations of Tesla cars in Sweden incr...,https://www.reuters.com/resizer/v2/JCBWEPODXJO...,Reuters,General,Reuters,"[{'topic': 'earnings', 'relevance_score': '0.9...",0.436517,Bullish,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."
4,Tesla registrations in Sweden rise 71% year-on...,https://uk.finance.yahoo.com/news/tesla-regist...,20260601T070900,[Reuters],Tesla registrations in Sweden increased by 71%...,https://s.yimg.com/ny/api/res/1.2/4D6TZgQeWqF1...,Yahoo Finance UK,General,Yahoo Finance UK,"[{'topic': 'earnings', 'relevance_score': '0.8...",0.430209,Bullish,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."


# Convert to DataFrame

In [4]:
print("Shape:", api_raw_df.shape)
print("Columns:")
print(api_raw_df.columns)

display(api_raw_df.head())

Shape: (895, 13)
Columns:
Index(['title', 'url', 'time_published', 'authors', 'summary', 'banner_image',
       'source', 'category_within_source', 'source_domain', 'topics',
       'overall_sentiment_score', 'overall_sentiment_label',
       'ticker_sentiment'],
      dtype='object')


,title,url,time_published,authors,summary,banner_image,source,category_within_source,source_domain,topics,overall_sentiment_score,overall_sentiment_label,ticker_sentiment
0,Why a Tesla and SpaceX Merger Would Not Be Gre...,https://finance.yahoo.com/markets/stocks/artic...,20260601T053842,[Lee Samaha],"A potential merger between Tesla and SpaceX, t...",https://s.yimg.com/ny/api/res/1.2/_bkOL8R5YV65...,Yahoo Finance,General,Yahoo Finance,"[{'topic': 'mergers_and_acquisitions', 'releva...",-0.087541,Neutral,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."
1,Will axing this 174-year-old brand boost Lloyd...,https://uk.finance.yahoo.com/news/axing-174-ol...,20260601T055100,[Royston Wild],"Lloyds' share price has performed strongly, an...",https://s.yimg.com/ny/api/res/1.2/rNX7rgx_aDgQ...,Yahoo Finance UK,General,Yahoo Finance UK,"[{'topic': 'finance', 'relevance_score': '0.92...",0.234030,Somewhat-Bullish,"[{'ticker': 'LYG', 'relevance_score': '1.00000..."
2,TSLA Stock Slips Overnight: SpaceX IPO Hype Is...,https://www.tradingview.com/news/stocktwits:fe...,20260601T060927,[],"A former Lehman Brothers trader, Larry McDonal...",None,TradingView,General,TradingView,"[{'topic': 'ipo', 'relevance_score': '0.930485...",-0.119459,Neutral,"[{'ticker': 'TSDD', 'relevance_score': '0.9448..."
3,Tesla registrations in Sweden rise 71% year-on...,https://www.reuters.com/business/tesla-registr...,20260601T070900,[Reuters],New registrations of Tesla cars in Sweden incr...,https://www.reuters.com/resizer/v2/JCBWEPODXJO...,Reuters,General,Reuters,"[{'topic': 'earnings', 'relevance_score': '0.9...",0.436517,Bullish,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."
4,Tesla registrations in Sweden rise 71% year-on...,https://uk.finance.yahoo.com/news/tesla-regist...,20260601T070900,[Reuters],Tesla registrations in Sweden increased by 71%...,https://s.yimg.com/ny/api/res/1.2/4D6TZgQeWqF1...,Yahoo Finance UK,General,Yahoo Finance UK,"[{'topic': 'earnings', 'relevance_score': '0.8...",0.430209,Bullish,"[{'ticker': 'TSLA', 'relevance_score': '1.0000..."


# Select Relevant Columns

In [5]:
selected_columns = [
    "title",
    "url",
    "time_published",
    "summary",
    "source",
    "overall_sentiment_score",
    "overall_sentiment_label",
    "ticker_sentiment"
]

api_cleaned_df = api_raw_df[selected_columns].copy()

def extract_tsla_field(value, field):
    """Extract TSLA-specific values from Alpha Vantage ticker_sentiment."""
    try:
        ticker_items = ast.literal_eval(value) if isinstance(value, str) else value
    except Exception:
        ticker_items = []

    if not isinstance(ticker_items, list):
        return None

    for item in ticker_items:
        if item.get("ticker") == "TSLA":
            return item.get(field)
    return None

api_cleaned_df["tsla_relevance_score"] = api_cleaned_df["ticker_sentiment"].apply(
    lambda value: extract_tsla_field(value, "relevance_score")
)
api_cleaned_df["tsla_sentiment_score"] = api_cleaned_df["ticker_sentiment"].apply(
    lambda value: extract_tsla_field(value, "ticker_sentiment_score")
)
api_cleaned_df["tsla_sentiment_label"] = api_cleaned_df["ticker_sentiment"].apply(
    lambda value: extract_tsla_field(value, "ticker_sentiment_label")
)
api_cleaned_df = api_cleaned_df.drop(columns=["ticker_sentiment"])

display(api_cleaned_df.head())

,title,url,time_published,summary,source,overall_sentiment_score,overall_sentiment_label,tsla_relevance_score,tsla_sentiment_score,tsla_sentiment_label
0,Why a Tesla and SpaceX Merger Would Not Be Gre...,https://finance.yahoo.com/markets/stocks/artic...,20260601T053842,"A potential merger between Tesla and SpaceX, t...",Yahoo Finance,-0.087541,Neutral,1.000000,-0.386958,Bearish
1,Will axing this 174-year-old brand boost Lloyd...,https://uk.finance.yahoo.com/news/axing-174-ol...,20260601T055100,"Lloyds' share price has performed strongly, an...",Yahoo Finance UK,0.234030,Somewhat-Bullish,0.588440,-0.520351,Bearish
2,TSLA Stock Slips Overnight: SpaceX IPO Hype Is...,https://www.tradingview.com/news/stocktwits:fe...,20260601T060927,"A former Lehman Brothers trader, Larry McDonal...",TradingView,-0.119459,Neutral,0.940547,-0.201928,Somewhat-Bearish
3,Tesla registrations in Sweden rise 71% year-on...,https://www.reuters.com/business/tesla-registr...,20260601T070900,New registrations of Tesla cars in Sweden incr...,Reuters,0.436517,Bullish,1.000000,0.447963,Bullish
4,Tesla registrations in Sweden rise 71% year-on...,https://uk.finance.yahoo.com/news/tesla-regist...,20260601T070900,Tesla registrations in Sweden increased by 71%...,Yahoo Finance UK,0.430209,Bullish,1.000000,0.412540,Bullish


# Clean Data

In [6]:
api_cleaned_df = api_cleaned_df.dropna(subset=["title", "url", "time_published", "tsla_relevance_score"])

api_cleaned_df["title"] = api_cleaned_df["title"].astype(str).str.strip()
api_cleaned_df["url"] = api_cleaned_df["url"].astype(str).str.strip()
api_cleaned_df["summary"] = api_cleaned_df["summary"].fillna("").astype(str).str.strip()
api_cleaned_df["source"] = api_cleaned_df["source"].fillna("").astype(str).str.strip()

api_cleaned_df["time_published"] = pd.to_datetime(
    api_cleaned_df["time_published"],
    format="%Y%m%dT%H%M%S",
    errors="coerce"
)
api_cleaned_df["date"] = api_cleaned_df["time_published"].dt.date

numeric_columns = ["overall_sentiment_score", "tsla_relevance_score", "tsla_sentiment_score"]
for column in numeric_columns:
    api_cleaned_df[column] = pd.to_numeric(api_cleaned_df[column], errors="coerce")

api_cleaned_df = api_cleaned_df.dropna(subset=["time_published", "date"])
api_cleaned_df = api_cleaned_df.drop_duplicates(subset=["url"])
api_cleaned_df = api_cleaned_df.sort_values("time_published", ascending=False)

print("Cleaned API rows:", len(api_cleaned_df))
display(api_cleaned_df.head())

Cleaned API rows: 895


,title,url,time_published,summary,source,overall_sentiment_score,overall_sentiment_label,tsla_relevance_score,tsla_sentiment_score,tsla_sentiment_label,date
894,New Tesla sales in Spain rise 5.6% year-on-yea...,https://www.reuters.com/business/retail-consum...,2026-07-01 10:59:50,Tesla's new car sales in Spain increased by 5....,Reuters,0.483065,Bullish,1.000000,0.495760,Bullish,2026-07-01
893,Top Gold Stocks as Franco-Nevada Expands Royal...,https://kalkinemedia.com/ca/stocks/gold/top-go...,2026-07-01 10:12:54,"Franco-Nevada, a constituent of the S&P/TSX 60...",Kalkine Media,0.221353,Somewhat-Bullish,0.612586,0.124234,Neutral,2026-07-01
892,Tesla's Cybercab is now testing without a stee...,https://www.statesman.com/business/article/tes...,2026-07-01 10:05:51,Tesla is advancing the testing of its Cybercab...,Austin American-Statesman,0.123753,Neutral,1.000000,0.132317,Neutral,2026-07-01
891,Apple CEO Tim Cook Flags 'Extreme Shortage' Wh...,https://www.benzinga.com/etfs/sector-etfs/26/0...,2026-07-01 07:43:59,"Apple CEO Tim Cook has highlighted an ""extreme...",Benzinga,0.311961,Somewhat-Bullish,0.749420,0.141140,Neutral,2026-07-01
890,General Mills stock rises 4% on earnings beat ...,https://www.investing.com/news/earnings/genera...,2026-07-01 07:23:00,General Mills (NYSE:GIS) reported fourth-quart...,Investing.com,0.236763,Somewhat-Bullish,0.585537,0.224878,Somewhat-Bullish,2026-07-01


# Check Missing Values

In [7]:
api_cleaned_df.isnull().sum()

title                      0
url                        0
time_published             0
summary                    0
source                     0
overall_sentiment_score    0
overall_sentiment_label    0
tsla_relevance_score       0
tsla_sentiment_score       0
tsla_sentiment_label       0
date                       0
dtype: int64

# Save Data

In [8]:
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

api_raw_df.to_csv("../data/raw/alpha_vantage_tesla_news_raw.csv", index=False)
api_cleaned_df.to_csv("../data/processed/alpha_vantage_tesla_news_cleaned.csv", index=False)

print("Saved raw data to ../data/raw/alpha_vantage_tesla_news_raw.csv")
print("Saved cleaned data to ../data/processed/alpha_vantage_tesla_news_cleaned.csv")
print("Final cleaned shape:", api_cleaned_df.shape)

Saved raw data to ../data/raw/alpha_vantage_tesla_news_raw.csv
Saved cleaned data to ../data/processed/alpha_vantage_tesla_news_cleaned.csv
Final cleaned shape: (895, 11)


# Quick Analysis

In [9]:
print("Date range:")
print(api_cleaned_df["date"].min(), "to", api_cleaned_df["date"].max())

print("\nOverall sentiment label counts:")
print(api_cleaned_df["overall_sentiment_label"].value_counts())

print("\nTSLA-specific sentiment label counts:")
print(api_cleaned_df["tsla_sentiment_label"].value_counts())

print("\nAverage overall sentiment score:")
print(api_cleaned_df["overall_sentiment_score"].mean())

print("\nAverage TSLA-specific sentiment score:")
print(api_cleaned_df["tsla_sentiment_score"].mean())


Date range:
2026-06-01 to 2026-07-01

Overall sentiment label counts:
overall_sentiment_label
Neutral             369
Somewhat-Bullish    305
Somewhat-Bearish    102
Bullish              80
Bearish              39
Name: count, dtype: int64

TSLA-specific sentiment label counts:
tsla_sentiment_label
Neutral             355
Somewhat-Bullish    266
Somewhat-Bearish    126
Bullish              82
Bearish              66
Name: count, dtype: int64

Average overall sentiment score:
0.08254948603351955

Average TSLA-specific sentiment score:
0.06624386927374301


# Short Conclusion

This notebook collected Tesla-related financial news from the Alpha Vantage News & Sentiment
API for the period 2026-06-01 to 2026-07-01, using the time_from/time_to parameters to
cover the full month instead of only the most recent articles.

The raw API response was cleaned and reduced to the most relevant columns for the project. A
key part of this step was extracting TSLA-specific relevance and sentiment values from the
ticker_sentiment field, since the overall article sentiment can differ from the sentiment
expressed specifically about Tesla within the same article.

The final cleaned dataset contains 895 news articles, spanning the entire overlap period with
the Tesla stock data. Both overall article sentiment (average score: 0.083, mostly Neutral to
Somewhat-Bullish) and TSLA-specific sentiment (average score: 0.066) were retained. Compared to
the earlier version of this pipeline, which only captured 3 days of news, this broader time
window now provides enough coverage to meaningfully merge with the daily stock data across all
21 overlapping trading days in June.

The cleaned API dataset is saved as alpha_vantage_tesla_news_cleaned.csv and is used later
for merging with the stock data and scraped news data.